In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from itertools import combinations
from sklearn.metrics import mean_squared_error

# Setting the random seed
np.random.seed(42)



In [2]:
# a. Generating the data
p = 20
n = 1000
dat = np.random.randn(n, p)
beta = np.zeros(p)
beta[0:4] = [5, 4, 2, 7]
y = np.dot(dat, beta) + np.random.randn(n)
dat = pd.DataFrame(dat, columns=[f'b{i+1}' for i in range(p)])
dat['y'] = y



In [3]:
# b. Splitting the data
train = dat.iloc[0:100]
test = dat.iloc[100:1000]



In [4]:
# c. Best subset selection
def fit_best_subset(X, y, k):
    models = list(combinations(X.columns, k))
    best_mse = float('inf')
    best_model = None
    
    for model in models:
        lm = LinearRegression().fit(X[list(model)], y)
        mse = mean_squared_error(y, lm.predict(X[list(model)]))
        if mse < best_mse:
            best_mse = mse
            best_model = model
    
    return best_model

train_mse = []
for i in range(1, p + 1):
    model = fit_best_subset(train.drop(columns=['y']), train['y'], i)
    lm = LinearRegression().fit(train[list(model)], train['y'])
    mse = mean_squared_error(train['y'], lm.predict(train[list(model)]))
    train_mse.append(mse)

# Plotting the training MSE
import matplotlib.pyplot as plt
plt.plot(range(1, p + 1), train_mse, marker='o')
plt.ylabel("MSE")
plt.xlabel("Model Size")
plt.show()


In [ ]:

# d. Testing the models on test set
test_mse = []
for i in range(1, p + 1):
    model = fit_best_subset(train.drop(columns=['y']), train['y'], i)
    lm = LinearRegression().fit(train[list(model)], train['y'])
    mse = mean_squared_error(test['y'], lm.predict(test[list(model)]))
    test_mse.append(mse)

# Plotting the test MSE
plt.plot(range(1, p + 1), test_mse, marker='o', color='r')
plt.ylabel("MSE")
plt.xlabel("Model Size")
plt.show()


In [ ]:

# e. Minimum MSE model size
min_mse_model_size = np.argmin(test_mse) + 1
print(f"The min test MSE is found when model size is {min_mse_model_size}.")


In [ ]:

# f. Coefficient comparison
best_model = fit_best_subset(train.drop(columns=['y']), train['y'], min_mse_model_size)
lm_best = LinearRegression().fit(train[list(best_model)], train['y'])
coeffs = lm_best.coef_
print("Estimated Coefficients:", coeffs)


In [ ]:

# g. Plotting coefficient error
errors = []
for i in range(1, p + 1):
    model = fit_best_subset(train.drop(columns=['y']), train['y'], i)
    lm = LinearRegression().fit(train[list(model)], train['y'])
    est_coeffs = np.zeros(p)
    true_coeffs = beta.copy()
    for idx, col in enumerate(model):
        j = int(col[1:]) - 1
        est_coeffs[j] = lm.coef_[idx]
    errors.append(np.sqrt(np.sum((true_coeffs - est_coeffs)**2)))

plt.plot(range(1, p + 1), errors, marker='o', color='b')
plt.ylabel("Mean squared coefficient error")
plt.xlabel("Model Size")
plt.show()
